In [1]:
# Cell 1: Load the merged dataframe produced in 01_eda.ipynb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet('../data/processed/df_merged.parquet')
print(f"Loaded: {df.shape}")
print(f"isFraud=1: {df['isFraud'].sum():,} ({df['isFraud'].mean()*100:.2f}%)")

Loaded: (590540, 436)
isFraud=1: 20,663 (3.50%)


In [2]:
# Cell 2: Signal 1 — device/identity change detection
# A sudden change in DeviceInfo for the same card is a strong ATO indicator
# Attacker uses different hardware/browser after credential theft

df = df.sort_values(['uid', 'TransactionDT']).reset_index(drop=True)

df['prev_device'] = df.groupby('uid')['DeviceInfo'].shift(1)

df['device_changed'] = (
    df['DeviceInfo'].notna() &
    df['prev_device'].notna() &
    (df['DeviceInfo'] != df['prev_device'])
).astype(int)

# id_31: proxy/anonymity network signal
df['is_proxy'] = df['id_31'].astype(str).str.contains(
    'proxy|anonymous|vpn', case=False, na=False
).astype(int)

df['signal_device'] = (
    (df['device_changed'] == 1) | (df['is_proxy'] == 1)
).astype(int)

print("=== Signal 1: Device/Identity ===")
print(f"Device changed    : {df['device_changed'].sum():,}")
print(f"Proxy/anonymous   : {df['is_proxy'].sum():,}")
print(f"signal_device = 1 : {df['signal_device'].sum():,}")
print(f"\nFraud rate where signal_device=1 : "
      f"{df[df['signal_device']==1]['isFraud'].mean()*100:.2f}%")
print(f"Fraud rate where signal_device=0 : "
      f"{df[df['signal_device']==0]['isFraud'].mean()*100:.2f}%")

=== Signal 1: Device/Identity ===
Device changed    : 32,463
Proxy/anonymous   : 0
signal_device = 1 : 32,463

Fraud rate where signal_device=1 : 7.08%
Fraud rate where signal_device=0 : 3.29%


In [3]:
# Cell 3: Signal 2 — session behaviour anomaly via id_01–id_11
# These columns encode login attempts, failed logins, session duration etc.
# An attacker's session profile deviates significantly from the legitimate user's history

id_cols = [f'id_0{i}' for i in range(1, 10)] + ['id_10', 'id_11']
id_cols = [c for c in id_cols if c in df.columns]

def session_anomaly(group, cols, threshold=2.0):
    """
    For each transaction in a UID group, check if any id column
    deviates more than `threshold` standard deviations from that
    UID's historical mean. Returns a binary anomaly flag.
    """
    result = pd.Series(False, index=group.index)
    for col in cols:
        col_vals = group[col].dropna()
        if len(col_vals) < 3:
            continue
        mu    = col_vals.mean()
        sigma = col_vals.std()
        if sigma == 0:
            continue
        z_scores = (group[col] - mu) / sigma
        result = result | (z_scores.abs() > threshold).fillna(False)
    return result.astype(int)

df['signal_session'] = df.groupby('uid', group_keys=False).apply(
    lambda g: session_anomaly(g, id_cols)
)

print("=== Signal 2: Session Anomaly ===")
print(f"signal_session = 1 : {df['signal_session'].sum():,}")
print(f"\nFraud rate where signal_session=1 : "
      f"{df[df['signal_session']==1]['isFraud'].mean()*100:.2f}%")
print(f"Fraud rate where signal_session=0 : "
      f"{df[df['signal_session']==0]['isFraud'].mean()*100:.2f}%")

=== Signal 2: Session Anomaly ===
signal_session = 1 : 23,915

Fraud rate where signal_session=1 : 11.55%
Fraud rate where signal_session=0 : 3.16%


In [4]:
# Cell 4: Signal 3 — transaction velocity anomaly
# Multiple transactions within a short window suggest automated/scripted ATO activity
# Legitimate users rarely make 2+ transactions within 2 hours on the same card

df['prev_dt'] = df.groupby('uid')['TransactionDT'].shift(1)
df['time_since_prev'] = df['TransactionDT'] - df['prev_dt']

# Flag transactions occurring less than 2 hours (7200 seconds) after the previous one
df['signal_velocity'] = (
    df['time_since_prev'].notna() &
    (df['time_since_prev'] < 7200)
).astype(int)

print("=== Signal 3: Velocity Anomaly ===")
print(f"signal_velocity = 1 : {df['signal_velocity'].sum():,}")
print(f"\nFraud rate where signal_velocity=1 : "
      f"{df[df['signal_velocity']==1]['isFraud'].mean()*100:.2f}%")
print(f"Fraud rate where signal_velocity=0 : "
      f"{df[df['signal_velocity']==0]['isFraud'].mean()*100:.2f}%")

=== Signal 3: Velocity Anomaly ===
signal_velocity = 1 : 185,673

Fraud rate where signal_velocity=1 : 5.00%
Fraud rate where signal_velocity=0 : 2.81%


In [5]:
# Cell 5: Combine signals — ATO proxy label requires at least 2 of 3 signals
# Single signals produce too many false positives (e.g. legitimate device upgrade)
# Requiring 2+ signals significantly improves precision while maintaining recall

df['signal_count'] = (
    df['signal_device'] +
    df['signal_session'] +
    df['signal_velocity']
)

df['ato_proxy'] = (
    (df['isFraud'] == 1) &
    (df['signal_count'] >= 2)
).astype(int)

# Distribution analysis
total_fraud = df['isFraud'].sum()
total_ato   = df['ato_proxy'].sum()

print("=== ATO Proxy Label Distribution ===")
print(f"Total isFraud=1          : {total_fraud:,}")
print(f"ato_proxy=1              : {total_ato:,} ({total_ato/total_fraud*100:.1f}% of fraud)")
print(f"ato_proxy=0 (other fraud): {total_fraud - total_ato:,}")
print(f"ato_proxy rate in dataset: {df['ato_proxy'].mean()*100:.3f}%")

# Signal combination breakdown
print("\n=== Signal count distribution among isFraud=1 ===")
print(df[df['isFraud']==1]['signal_count'].value_counts().sort_index())

=== ATO Proxy Label Distribution ===
Total isFraud=1          : 20,663
ato_proxy=1              : 2,362 (11.4% of fraud)
ato_proxy=0 (other fraud): 18,301
ato_proxy rate in dataset: 0.400%

=== Signal count distribution among isFraud=1 ===
0    8946
1    9355
2    2094
3     268
Name: signal_count, dtype: int64


In [6]:
# Cell 6: Save labeled dataframe and print methodology summary
df.to_parquet('../data/processed/df_ato_labeled.parquet', index=False)
print("Saved: data/processed/df_ato_labeled.parquet")

print("\n=== PROXY LABEL METHODOLOGY SUMMARY ===")
print("""
ATO proxy labels were constructed using three behavioural signals
derived from the IEEE-CIS dataset, following Alghofaili et al. (2020):

  Signal 1 — Device/identity change:
    DeviceInfo changed between consecutive transactions on the same card,
    or id_31 indicates proxy/anonymous network usage.

  Signal 2 — Session anomaly:
    Any id_01–id_11 value deviates >2 standard deviations from the
    card's historical mean (z-score based).

  Signal 3 — Velocity anomaly:
    Two or more transactions on the same card within a 2-hour window.

Labelling rule:
    ato_proxy = 1  if  isFraud=1  AND  signal_count >= 2
    ato_proxy = 0  otherwise

Limitation:
    No ground-truth ATO label exists in the dataset. This proxy
    methodology is a standard approach in the literature when
    sub-category fraud labels are unavailable.
""")

print(f"Next step -> 03_feature_eng.ipynb")

Saved: data/processed/df_ato_labeled.parquet

=== PROXY LABEL METHODOLOGY SUMMARY ===

ATO proxy labels were constructed using three behavioural signals
derived from the IEEE-CIS dataset, following Alghofaili et al. (2020):

  Signal 1 — Device/identity change:
    DeviceInfo changed between consecutive transactions on the same card,
    or id_31 indicates proxy/anonymous network usage.

  Signal 2 — Session anomaly:
    Any id_01–id_11 value deviates >2 standard deviations from the
    card's historical mean (z-score based).

  Signal 3 — Velocity anomaly:
    Two or more transactions on the same card within a 2-hour window.

Labelling rule:
    ato_proxy = 1  if  isFraud=1  AND  signal_count >= 2
    ato_proxy = 0  otherwise

Limitation:
    No ground-truth ATO label exists in the dataset. This proxy
    methodology is a standard approach in the literature when
    sub-category fraud labels are unavailable.

Next step -> 03_feature_eng.ipynb
